# 01 — Hello, ML: Breast Cancer Classification
Binary classification (benign vs malignant) using a small built-in dataset. Run each cell in order. Designed for Google Colab.

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, RocCurveDisplay
from sklearn.metrics import ConfusionMatrixDisplay
import warnings; warnings.filterwarnings('ignore')
RANDOM_STATE = 42

In [ ]:
# Load data and peek
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target  # 0 malignant, 1 benign
display(X.head())
display(y.value_counts())

In [ ]:
# Split into train/test (hold out 20% for unbiased evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
# Build a pipeline: Standardize features -> Logistic Regression
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=RANDOM_STATE))
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
proba = pipe.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))
RocCurveDisplay.from_predictions(y_test, proba)
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.show()

In [ ]:
# Quick sanity check with cross-validation on the TRAIN set only
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
print("5-fold CV ROC-AUC (train subset) mean±sd:", cv_scores.mean(), "+/-", cv_scores.std())

**Notes**
- `stratify=y` keeps class balance in the split.
- Use ROC-AUC in addition to accuracy for binary classification.
- Fix `random_state` for reproducibility.